In [5]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

print(project_root)

c:\Users\SWS\Desktop\summer\projects\credit fraud


In [8]:
import pandas as pd

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier

from src.preprocessing import clean_data, PREPROCESSING_PIPELINES

In [10]:
# Clean data
df= pd.read_csv("../data/creditcard.csv")
df = clean_data(df)

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [11]:
model_configs = {
    "Logistic Regression": {
        "preprocessor": PREPROCESSING_PIPELINES["linear_time_cyclic_scaled"],
        "model": LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    },

    "Random Forest": {
        "preprocessor": PREPROCESSING_PIPELINES["tree_time_cyclic"],
        "model": RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    },

    "XGBoost": {
        "preprocessor": PREPROCESSING_PIPELINES["tree_time_cyclic"],
        "model": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            eval_metric="logloss",
            random_state=42
        )
    }
}

In [12]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "AUPRC": "average_precision",
    "ROC_AUC": "roc_auc",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1"
}

In [13]:
cv_results_list = []

for model_name, config in model_configs.items():

    pipeline = ImbPipeline([
        ("preprocessing", config["preprocessor"]),
        ("smote", SMOTE(random_state=42)),
        ("model", config["model"])
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results_list.append({
        "model": model_name,
        "AUPRC_mean": scores["test_AUPRC"].mean(),
        "AUPRC_std": scores["test_AUPRC"].std(),
        "ROC_AUC_mean": scores["test_ROC_AUC"].mean(),
        "Precision_mean": scores["test_Precision"].mean(),
        "Recall_mean": scores["test_Recall"].mean(),
        "F1_mean": scores["test_F1"].mean()
    })

cv_results_df = pd.DataFrame(cv_results_list)
cv_results_df = cv_results_df.sort_values(by="AUPRC_mean", ascending=False)

cv_results_df

,model,AUPRC_mean,AUPRC_std,ROC_AUC_mean,Precision_mean,Recall_mean,F1_mean
1,Random Forest,0.845952,0.031003,0.979749,0.910243,0.812175,0.856766
2,XGBoost,0.820857,0.025219,0.980177,0.577313,0.838596,0.682965
0,Logistic Regression,0.755184,0.025466,0.978831,0.054254,0.907298,0.102329


In [15]:
best_model_name = cv_results_df.iloc[0]["model"]
best_config = model_configs[best_model_name]

final_pipeline = ImbPipeline([
    ("preprocessing", best_config["preprocessor"]),
    ("smote", SMOTE(random_state=42)),
    ("model", best_config["model"])
])

final_pipeline.fit(X_train, y_train)

y_pred = final_pipeline.predict(X_test)
y_proba = final_pipeline.predict_proba(X_test)[:, 1]

test_results = pd.DataFrame([{
    "model": best_model_name,
    "AUPRC": average_precision_score(y_test, y_proba),
    "ROC_AUC": roc_auc_score(y_test, y_proba),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred)
}])

test_results

,model,AUPRC,ROC_AUC,Precision,Recall,F1
0,Random Forest,0.80429,0.961625,0.935065,0.757895,0.837209


In [17]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    final_pipeline,
    "../models/best_model.pkl"
)

print("Model saved successfully.")

Model saved successfully.


## Model Comparison Results

Three machine learning models were evaluated using 5-fold stratified cross-validation on the training set. The primary evaluation metric was **AUPRC (Area Under the Precision-Recall Curve)**, which is particularly suitable for highly imbalanced fraud detection datasets.

| Model | AUPRC | ROC-AUC | Precision | Recall | F1 |
|---------|---------|---------|---------|---------|---------|
| Random Forest | **0.846** | 0.980 | **0.910** | 0.812 | **0.857** |
| XGBoost | 0.821 | **0.980** | 0.577 | 0.839 | 0.683 |
| Logistic Regression | 0.755 | 0.979 | 0.054 | **0.907** | 0.102 |

### Cross-Validation Analysis

The **Random Forest** model achieved the highest AUPRC (0.846), indicating the best overall ability to identify fraudulent transactions while minimizing false positives. It also obtained the highest precision (91.0%) and F1-score (85.7%), demonstrating a strong balance between fraud detection and false alarm reduction.

The **XGBoost** model produced competitive results with an AUPRC of 0.821 and the highest ROC-AUC score. However, its precision was considerably lower than Random Forest, indicating a larger number of false positive predictions.

The **Logistic Regression** baseline achieved the highest recall (90.7%), meaning it successfully detected most fraudulent transactions. However, its extremely low precision (5.4%) indicates that a large proportion of transactions flagged as fraud were actually legitimate transactions. This behavior is typical when using linear models on highly imbalanced datasets.

Based on the cross-validation results, **Random Forest was selected as the best-performing model** for further evaluation and optimization.

---

## Final Test Set Evaluation

After selecting Random Forest as the best model, it was retrained on the entire training dataset and evaluated on the unseen test set.

| Metric | Score |
|----------|----------|
| AUPRC | 0.804 |
| ROC-AUC | 0.962 |
| Precision | 0.935 |
| Recall | 0.758 |
| F1 | 0.837 |

### Test Set Analysis

The model maintained strong performance on unseen data, achieving an AUPRC of 0.804 and a ROC-AUC of 0.962. These results confirm that the model generalizes well and is not significantly overfitting the training data.

The precision of 93.5% indicates that the vast majority of transactions flagged as fraudulent were indeed fraud cases. This is particularly desirable in a real-world fraud detection system because it minimizes unnecessary investigations and customer disruptions caused by false alarms.

The recall of 75.8% shows that the model successfully detected approximately three out of every four fraudulent transactions. While lower than the recall observed with Logistic Regression, this reduction is compensated by a substantial increase in precision.

Overall, the Random Forest model provides the best trade-off between fraud detection capability and false positive reduction. Consequently, it was retained as the final candidate model for hyperparameter tuning and threshold optimization in the subsequent phases of the project.